# Admissions Essay Stylometrics

> **Data sensitivity:** This notebook uses restricted admissions data.  
> Run locally only â€” do not commit `essay_assessment/data/` files to the repository.

Stylometrics asks: *how* is something written, independent of *what* it says.  
We extract six families of features across **three corpora** (real, synthetic, freeform)
and examine how they vary by corpus and relate to reviewer scores.

| # | Feature family | Key metrics |
|---|---|---|
| 1 | **Lexical richness** | MTLD, HD-D, vocabulary size, rare-word rate |
| 2 | **Syntactic complexity** | Dependency tree depth, clause count, subordination ratio |
| 3 | **Surface / length** | Sentence length (mean, std, CV), word length, paragraph count |
| 4 | **Discourse markers** | Density of connectives (contrast, causal, additive, temporal) |
| 5 | **Figurative & decorative language** | Simile density, intensifiers, decorative adjectives, hollow phrases, rhetorical questions, anaphora |
| 6 | **Style clustering** | Unsupervised writer-profile clusters; score distribution per cluster; corpus separation in PCA space |

**Three corpora loaded:** `real` (admissions essays with reviewer scores), `synthetic`, `freeform`


## Dependencies

In [ ]:
# Uncomment and run once if packages are missing
!pip install spacy scikit-learn pandas matplotlib seaborn lexicalrichness
!python -m spacy download en_core_web_sm

## Imports & config

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from lexicalrichness import LexicalRichness
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

 # Anchor all paths to the notebook's own directory so they resolve correctly
# regardless of what directory the Jupyter kernel happens to start in.
try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).parent   # VS Code injects this
except NameError:
    NOTEBOOK_DIR = Path(__file__).parent              # plain script fallback

CORPORA = {
    'real':      NOTEBOOK_DIR / '../data/real/essays.xlsx',
    'synthetic': NOTEBOOK_DIR / '../data/synthetic/synthetic_essays.csv',
    'freeform':  NOTEBOOK_DIR / '../data/freeform/freeform_essays.csv',
}
SAMPLE_N     = None   # per corpus; set None for full run
RANDOM_STATE = 42
CORPUS_PALETTE = {'real': '#4C72B0', 'synthetic': '#DD8452', 'freeform': '#55A868'}


## Load essays

In [ ]:
DUMMY_ESSAYS = [
    ('Growing up in a small town, I never imagined standing before a crowd. '
     'I struggled with anxiety for years. But joining debate changed me. '
     'I learned my voice matters. Now I want to study psychology to help others find theirs.',
     3.5),
    ('My mother works two jobs. Every morning I watch her leave before sunrise. '
     'That image drives everything I do. I maintain a 4.0 GPA not for myself, but for her. '
     'I intend to become an engineer and give her the life she deserves.',
     4.0),
    ('I have always believed science can solve humanity\'s greatest problems. '
     'Studies show renewable energy reduces emissions by up to 70 percent. '
     'I conducted my own solar panel efficiency research. '
     'The data confirmed that innovation, not incremental change, is required.',
     3.0),
    ('Some say art is a luxury. I disagree. After Hurricane Maria devastated my community, '
     'art became our therapy. I organized murals, open mics, and poetry slams. '
     'Counter to those who dismiss the humanities, I argue they are essential infrastructure.',
     4.5),
    ('I want to be a doctor. I shadowed a cardiologist for 80 hours last summer. '
     'I observed three open-heart surgeries. Every case reinforced my commitment. '
     'Medicine is not just a career; it is a calling I cannot ignore.',
     3.5),
    ('Leadership is not about authority; it is about service. '
     'As student council president I launched a mental health awareness week. '
     'Attendance exceeded 600 students. I am proud of that enduring impact.',
     4.0),
    ('When I was eight my grandmother taught me to cook traditional Haitian dishes. '
     'Those afternoons shaped my identity more than any classroom ever could. '
     'Food is culture, memory, and love. I plan to study nutrition science through that lens.',
     3.0),
    ('The first time I coded a working program, I felt a joy I had never experienced before. '
     'I subsequently built a tutoring app now used by 300 students in my district. '
     'Technology, I have come to realize, is the most scalable form of empathy available.',
     4.5),
]

# Maps alternate column names found in real data files â†’ canonical names used by this notebook
_COL_ALIASES = {
    'essay':      ['essay', 'Essay Response', 'essay_text', 'text', 'Essay', 'response'],
    'score':      ['score', 'Essay Score', 'proxy_score', 'Score', 'rating', 'Rating'],
    'student_id': ['student_id', 'Campus ID', 'id', 'ID', 'student id', 'StudentID'],
}

def _normalize_df(raw_df, corpus_name):
    """Ensure essay / score / student_id / corpus columns are present and consistent."""
    df_out = raw_df.copy()

    # Remap alternate column names to canonical names
    rename_map = {}
    for canonical, aliases in _COL_ALIASES.items():
        if canonical not in df_out.columns:
            for alias in aliases:
                if alias in df_out.columns:
                    rename_map[alias] = canonical
                    break
    if rename_map:
        df_out = df_out.rename(columns=rename_map)

    if 'essay' not in df_out.columns:
        raise ValueError(f'{corpus_name}: no "essay" column. Found: {list(df_out.columns)}')

    # Normalize score column
    if 'score' not in df_out.columns:
        df_out['score'] = float('nan')

    # Unique student_id per corpus so merge keys never collide
    if 'student_id' not in df_out.columns:
        df_out['student_id'] = range(len(df_out))
    df_out['student_id'] = corpus_name[:3] + '_' + df_out['student_id'].astype(str)

    df_out['corpus'] = corpus_name
    return df_out[['student_id', 'corpus', 'essay', 'score']]


def load_all_corpora(corpora=CORPORA, sample_n=SAMPLE_N):
    frames = []
    for corpus_name, path in corpora.items():
        path = Path(path)
        if not path.exists():
            print(f'  [skip] {corpus_name}: {path} not found')
            continue
        raw = pd.read_excel(path) if path.suffix == '.xlsx' else pd.read_csv(path)
        if path.suffix == '.xlsx':
            raw = raw.dropna(subset=['Essay Score', 'Essay Response'])
        frame = _normalize_df(raw, corpus_name)
        if sample_n and len(frame) > sample_n:
            frame = frame.sample(sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
        frames.append(frame)
        print(f'  {corpus_name}: {len(frame):,} essays  ({path.name})')

    if not frames:
        print('No corpora found â€” using short dummy "real" dataset.')
        dummy = pd.DataFrame({
            'student_id': [f'rea_{i}' for i in range(1, len(DUMMY_ESSAYS) + 1)],
            'corpus':     'real',
            'essay':      [e for e, _ in DUMMY_ESSAYS],
            'score':      [s for _, s in DUMMY_ESSAYS],
        })
        frames.append(dummy)

    df = pd.concat(frames, ignore_index=True)
    print(f'\nTotal: {len(df):,} essays across {df["corpus"].nunique()} corpus/corpora')
    return df


df = load_all_corpora()
print(df.groupby('corpus')[['score']].describe().round(2))


## spaCy setup

In [ ]:
# Load once; reuse the same parsed docs across all feature extractors
nlp = spacy.load('en_core_web_sm')

essays = df['essay'].astype(str).tolist()
docs = list(tqdm(nlp.pipe(essays, batch_size=32), total=len(essays), desc='Parsing with spaCy'))
print(f'Done â€” {len(docs)} docs parsed.')


In [ ]:
# One-time cleanup: drop any _x / _y duplicate columns left over from earlier runs
# Safe to re-run â€” does nothing if the columns are already clean.
xy_cols = [c for c in df.columns if c.endswith('_x') or c.endswith('_y')]
if xy_cols:
    df = df.drop(columns=xy_cols)
    print(f'Dropped {len(xy_cols)} duplicate columns: {xy_cols}')
else:
    print('No _x/_y columns found â€” df is clean.')


---
## Section 1: Lexical Richness

Simple type-token ratio (TTR) is unreliable at variable text lengths â€” longer essays
mechanically score lower. We use length-stable alternatives instead:

| Metric | What it measures | Why it's better than TTR |
|---|---|---|
| **MTLD** | Mean length of sequential token runs maintaining a TTR â‰¥ 0.72 | Unaffected by text length |
| **HD-D** | Probability that a random sample of 42 words contains a new type | Length-independent |
| **Vocabulary size** | Raw unique lemma count | Context for other metrics |
| **Rare-word rate** | Fraction of tokens appearing only once in the essay | Proxy for advanced vocabulary |
| **Avg word length** | Mean character count per token (excl. punctuation) | Surface complexity proxy |

In [ ]:
def lexical_richness_features(doc):
    tokens = [t.text.lower() for t in doc if not t.is_punct and not t.is_space]
    lemmas = [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]

    if len(tokens) < 10:
        return {k: float('nan') for k in
                ['mtld', 'hdd', 'vocab_size', 'rare_word_rate', 'avg_word_length']}

    text_str = ' '.join(tokens)
    lex = LexicalRichness(text_str)

    freq = Counter(tokens)
    rare_rate = sum(1 for c in freq.values() if c == 1) / len(freq)
    avg_wlen  = np.mean([len(t) for t in tokens])

    return {
        'mtld':           lex.mtld(threshold=0.72),
        'hdd':            lex.hdd(draws=42),
        'vocab_size':     len(set(lemmas)),
        'rare_word_rate': rare_rate,
        'avg_word_length': avg_wlen,
    }

lex_rows = [
    {'student_id': row.student_id, **lexical_richness_features(doc)}
    for row, doc in tqdm(zip(df.itertuples(), docs), total=len(df), desc='Lexical richness')
]
lex_df = pd.DataFrame(lex_rows)
lex_features = ['mtld', 'hdd', 'vocab_size', 'rare_word_rate', 'avg_word_length']
df = df.drop(columns=[c for c in lex_features if c in df.columns])
df = df.merge(lex_df, on='student_id')

print('Lexical richness â€” mean by corpus:')
print(df.groupby('corpus')[lex_features].mean().round(3).to_string())


In [ ]:
lex_features = ['mtld', 'hdd', 'vocab_size', 'rare_word_rate', 'avg_word_length']

# Score correlation â€” real essays only (where scores are meaningful)
real_df = df[df['corpus'] == 'real'].dropna(subset=['score'] + lex_features)
if len(real_df) > 1:
    corrs = real_df[lex_features + ['score']].corr()['score'].drop('score').round(3)
    print('Pearson r with reviewer score (real essays only):')
    print(corrs)

# Corpus comparison: violin plots for all 5 features
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, feat in zip(axes, lex_features):
    valid = df.dropna(subset=[feat])
    sns.violinplot(
        data=valid, x='corpus', y=feat, ax=ax,
        palette=CORPUS_PALETTE, cut=0, inner='box',
        order=[c for c in CORPUS_PALETTE if c in valid['corpus'].unique()],
    )
    ax.set_title(feat.replace('_', ' ').upper())
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)
plt.suptitle('Lexical richness by corpus', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 2: Syntactic Complexity

Parsed dependency trees give us sentence-level structural information that
surface statistics miss entirely.

| Metric | How it's computed |
|---|---|
| **Mean parse depth** | Longest root-to-leaf path in each sentence's dependency tree, averaged across the essay |
| **Clause count** | Number of finite verbs (proxy for independent + subordinate clauses) |
| **Subordination ratio** | Fraction of clauses introduced by a subordinating conjunction (`mark` dep) |
| **Mean deps per token** | Average number of syntactic dependents per content word |

In [ ]:
def tree_depth(token):
    """Recursive depth of the subtree rooted at token."""
    if not list(token.children):
        return 0
    return 1 + max(tree_depth(c) for c in token.children)

def syntactic_features(doc):
    sents = list(doc.sents)
    if not sents:
        return {k: float('nan') for k in
                ['mean_parse_depth', 'clause_count', 'subordination_ratio', 'mean_deps_per_token']}

    depths       = [tree_depth(s.root) for s in sents]
    finite_verbs = [t for t in doc if t.pos_ == 'VERB' and t.morph.get('VerbForm') != ['Inf']]
    subord_marks = [t for t in doc if t.dep_ == 'mark']
    content_toks = [t for t in doc if t.pos_ in ('NOUN', 'VERB', 'ADJ', 'ADV')]
    deps_per_tok = (
        np.mean([len(list(t.children)) for t in content_toks])
        if content_toks else float('nan')
    )

    n_clauses = max(len(finite_verbs), 1)
    return {
        'mean_parse_depth':     float(np.mean(depths)),
        'clause_count':         n_clauses,
        'subordination_ratio':  len(subord_marks) / n_clauses,
        'mean_deps_per_token':  float(deps_per_tok),
    }

syn_rows = [
    {'student_id': row.student_id, **syntactic_features(doc)}
    for row, doc in tqdm(zip(df.itertuples(), docs), total=len(df), desc='Syntactic complexity')
]
syn_df = pd.DataFrame(syn_rows)
syn_features = ['mean_parse_depth', 'clause_count', 'subordination_ratio', 'mean_deps_per_token']
df = df.drop(columns=[c for c in syn_features if c in df.columns])
df = df.merge(syn_df, on='student_id')

print('Syntactic complexity â€” mean by corpus:')
print(df.groupby('corpus')[syn_features].mean().round(3).to_string())


In [ ]:
real_df = df[df['corpus'] == 'real'].dropna(subset=['score'] + syn_features)
if len(real_df) > 1:
    corrs = real_df[syn_features + ['score']].corr()['score'].drop('score').round(3)
    print('Pearson r with reviewer score (real essays only):')
    print(corrs)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, feat in zip(axes, syn_features):
    valid = df.dropna(subset=[feat])
    sns.violinplot(
        data=valid, x='corpus', y=feat, ax=ax,
        palette=CORPUS_PALETTE, cut=0, inner='box',
        order=[c for c in CORPUS_PALETTE if c in valid['corpus'].unique()],
    )
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)
plt.suptitle('Syntactic complexity by corpus', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 3: Surface & Length Features

These are cheap to compute and often surprisingly predictive. Coefficient of
variation (CV = std / mean) on sentence length captures *rhythmic variety* â€”
good writers tend to vary sentence length purposefully.

In [ ]:
def surface_features(text, doc):
    sents      = list(doc.sents)
    sent_lens  = [len([t for t in s if not t.is_space]) for s in sents]
    tokens     = [t for t in doc if not t.is_punct and not t.is_space]
    paragraphs = [p.strip() for p in str(text).split('\n') if p.strip()]

    if not sent_lens:
        return {k: float('nan') for k in
                ['n_sentences', 'n_tokens', 'n_paragraphs',
                 'mean_sent_len', 'std_sent_len', 'cv_sent_len',
                 'median_sent_len', 'max_sent_len']}

    mean_sl = float(np.mean(sent_lens))
    std_sl  = float(np.std(sent_lens))
    return {
        'n_sentences':    len(sents),
        'n_tokens':       len(tokens),
        'n_paragraphs':   max(len(paragraphs), 1),
        'mean_sent_len':  mean_sl,
        'std_sent_len':   std_sl,
        'cv_sent_len':    std_sl / mean_sl if mean_sl > 0 else float('nan'),
        'median_sent_len': float(np.median(sent_lens)),
        'max_sent_len':   float(max(sent_lens)),
    }

surf_rows = [
    {'student_id': row.student_id, **surface_features(row.essay, doc)}
    for row, doc in tqdm(zip(df.itertuples(), docs), total=len(df), desc='Surface features')
]
surf_df = pd.DataFrame(surf_rows)
surf_features = ['n_sentences', 'n_tokens', 'mean_sent_len', 'std_sent_len',
                 'cv_sent_len', 'median_sent_len']
df = df.drop(columns=[c for c in surf_features + ['n_paragraphs', 'max_sent_len'] if c in df.columns])
df = df.merge(surf_df, on='student_id')

print('Surface features â€” mean by corpus:')
print(df.groupby('corpus')[surf_features].mean().round(2).to_string())


In [ ]:
real_df = df[df['corpus'] == 'real'].dropna(subset=['score'] + surf_features)
if len(real_df) > 1:
    corrs = real_df[surf_features + ['score']].corr()['score'].drop('score').round(3)
    print('Pearson r with reviewer score (real essays only):')
    print(corrs)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()
for ax, feat in zip(axes, surf_features):
    valid = df.dropna(subset=[feat])
    sns.violinplot(
        data=valid, x='corpus', y=feat, ax=ax,
        palette=CORPUS_PALETTE, cut=0, inner='box',
        order=[c for c in CORPUS_PALETTE if c in valid['corpus'].unique()],
    )
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)
plt.suptitle('Surface features by corpus', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 4: Discourse Marker Density

Connectives signal how a writer links ideas. We count four functional categories
per 100 words and examine whether their density correlates with score.

| Category | Example markers |
|---|---|
| **Contrast** | however, although, but, yet, nevertheless, despite, whereas |
| **Causal** | because, therefore, thus, hence, consequently, so, since |
| **Additive** | furthermore, moreover, additionally, also, in addition, besides |
| **Temporal** | first, then, finally, subsequently, meanwhile, afterward |

In [ ]:
CONNECTIVES = {
    'contrast':  {'however', 'although', 'though', 'but', 'yet', 'nevertheless',
                  'nonetheless', 'despite', 'whereas', 'while', 'even though',
                  'on the other hand', 'in contrast', 'conversely'},
    'causal':    {'because', 'therefore', 'thus', 'hence', 'consequently',
                  'as a result', 'so', 'since', 'due to', 'accordingly',
                  'for this reason'},
    'additive':  {'furthermore', 'moreover', 'additionally', 'also', 'besides',
                  'in addition', 'likewise', 'similarly', 'and', 'as well as'},
    'temporal':  {'first', 'second', 'third', 'then', 'finally', 'subsequently',
                  'meanwhile', 'afterward', 'before', 'after', 'when',
                  'in the beginning', 'at last', 'eventually'},
}

def marker_density(text, n_tokens):
    """Count connective hits per 100 tokens for each category."""
    text_lower = text.lower()
    per_100    = 100 / max(n_tokens, 1)
    result = {}
    for cat, markers in CONNECTIVES.items():
        # Whole-phrase matching (handles multi-word markers)
        hits = sum(len(re.findall(r'\b' + re.escape(m) + r'\b', text_lower))
                   for m in markers)
        result[f'marker_{cat}'] = hits * per_100
    result['marker_total'] = sum(result.values())
    return result

marker_cols = ['marker_contrast', 'marker_causal', 'marker_additive',
               'marker_temporal', 'marker_total']
marker_rows = [
    {'student_id': row.student_id,
     **marker_density(row.essay, df.loc[df.student_id == row.student_id, 'n_tokens'].values[0])}
    for row in tqdm(df.itertuples(), total=len(df), desc='Discourse markers')
]
marker_df = pd.DataFrame(marker_rows)
df = df.drop(columns=[c for c in marker_cols if c in df.columns])
df = df.merge(marker_df, on='student_id')

print('Discourse marker density (per 100 tokens) â€” mean by corpus:')
print(df.groupby('corpus')[marker_cols].mean().round(3).to_string())


In [ ]:
real_df = df[df['corpus'] == 'real'].dropna(subset=['score'] + marker_cols)
if len(real_df) > 1:
    corrs = real_df[marker_cols + ['score']].corr()['score'].drop('score').round(3)
    print('Pearson r with reviewer score (real essays only):')
    print(corrs)

# Mean marker density per corpus per category
mean_by_corpus = df.groupby('corpus')[marker_cols[:-1]].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Grouped bar: mean density by category Ã— corpus
mean_by_corpus.T.plot(kind='bar', ax=axes[0], color=[CORPUS_PALETTE.get(c, 'gray') for c in mean_by_corpus.index], edgecolor='white')
axes[0].set_xticklabels([c.replace('marker_', '').title() for c in marker_cols[:-1]], rotation=0)
axes[0].set_ylabel('Mean per 100 tokens')
axes[0].set_title('Discourse marker density by category & corpus')
axes[0].legend(title='Corpus')

# Violin: total marker density by corpus
valid = df.dropna(subset=['marker_total'])
sns.violinplot(
    data=valid, x='corpus', y='marker_total', ax=axes[1],
    palette=CORPUS_PALETTE, cut=0, inner='box',
    order=[c for c in CORPUS_PALETTE if c in valid['corpus'].unique()],
)
axes[1].set_title('Total discourse marker density by corpus')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()


---
## Section 5: Figurative & Decorative Language

The hypothesis: LLM-generated essays lean on rhetorical "decoration" â€”
intensifiers, high-register evaluative adjectives, simile constructions,
and hollow clichÃ©d phrases â€” more than authentic student writing.

| Feature | What it captures |
|---|---|
| **Simile density** | "like a / as X as / as if" constructions per 100 tokens |
| **Intensifier density** | Degree amplifiers ("extremely", "profoundly", "absolutely"â€¦) per 100 tokens |
| **Decorative adj density** | High-register evaluative adjectives ("transformative", "profound"â€¦) per 100 tokens |
| **Hollow phrase density** | ClichÃ©d filler phrases ("at the end of the day", "delve into"â€¦) per 100 tokens |
| **Rhetorical question rate** | Interrogative sentences Ã· total sentences |
| **Anaphora proxy** | Repeated sentence-initial lemmas (normalised by sentence count) |
| **Composite** | Sum of density metrics â€” overall "rhetorical decoration" index |


In [ ]:
SIMILE_PATTERNS = [
    r'\blike\s+an?\b',        # like a / like an
    r'\bas\s+\w+\s+as\b',     # as X as (e.g. "as clear as")
    r'\bas\s+if\b',            # as if
    r'\bjust\s+as\b',          # just as
    r'\bsimilar\s+to\b',       # similar to
    r'\bakin\s+to\b',          # akin to
]

INTENSIFIERS = {
    'very', 'extremely', 'incredibly', 'absolutely', 'truly', 'deeply',
    'profoundly', 'remarkably', 'utterly', 'completely', 'entirely',
    'immensely', 'enormously', 'overwhelmingly', 'exceptionally',
    'undeniably', 'undoubtedly', 'certainly', 'definitely', 'literally',
    'so', 'quite', 'rather', 'really', 'terribly', 'awfully',
}

DECORATIVE_ADJ = {
    'transformative', 'profound', 'remarkable', 'extraordinary', 'pivotal',
    'invaluable', 'unprecedented', 'multifaceted', 'nuanced', 'intrinsic',
    'inherent', 'crucial', 'vital', 'essential', 'fundamental', 'impactful',
    'meaningful', 'powerful', 'insightful', 'inspiring', 'captivating',
    'compelling', 'dynamic', 'innovative', 'groundbreaking', 'seamless',
    'vibrant', 'empowering', 'holistic', 'robust', 'synergistic',
    'unparalleled', 'transcendent', 'quintessential', 'overarching',
    'poignant', 'enriching', 'fulfilling', 'rewarding',
}

# Phrases that are statistically overrepresented in LLM-generated text
HOLLOW_PHRASES = [
    "at the end of the day", "needless to say", "it goes without saying",
    "in today's world", "in a world where", "it is important to note",
    "it is worth noting", "throughout history", "without a doubt",
    "first and foremost", "last but not least", "in conclusion",
    "it is clear that", "it cannot be overstated", "more than ever",
    "in many ways", "a testament to", "not only", "in the grand scheme",
    "at its core", "at its heart", "deeply rooted", "stands as a testament",
    "delve into", "tapestry of", "in the realm of", "foster a sense of",
    "dive into", "showcases the importance", "it is worth mentioning",
]


def figurative_features(text, doc):
    text_lower = text.lower()
    sents      = list(doc.sents)
    n_sents    = max(len(sents), 1)
    tokens     = [t for t in doc if not t.is_punct and not t.is_space]
    n_tokens   = max(len(tokens), 1)
    per_100    = 100 / n_tokens

    # Simile density (regex)
    simile_hits = sum(len(re.findall(p, text_lower)) for p in SIMILE_PATTERNS)

    # Intensifier density â€” ADV or ADJ tagged by spaCy
    intensifier_hits = sum(
        1 for t in doc
        if t.lemma_.lower() in INTENSIFIERS and t.pos_ in ('ADV', 'ADJ')
    )

    # Decorative / evaluative adjective density â€” must be tagged ADJ
    decorative_hits = sum(
        1 for t in doc
        if t.lemma_.lower() in DECORATIVE_ADJ and t.pos_ == 'ADJ'
    )

    # Hollow phrase density (whole-phrase match)
    hollow_hits = sum(
        len(re.findall(r'\b' + re.escape(p) + r'\b', text_lower))
        for p in HOLLOW_PHRASES
    )

    # Rhetorical question rate
    rhetorical_q = sum(1 for s in sents if s.text.strip().endswith('?'))

    # Anaphora proxy: sentences sharing the same opening lemma
    first_lemmas = [
        s[0].lemma_.lower() for s in sents
        if len(s) > 0 and not s[0].is_punct
    ]
    freq = Counter(first_lemmas)
    anaphora = sum(c - 1 for c in freq.values() if c > 1) / n_sents

    composite = (simile_hits + intensifier_hits + decorative_hits + hollow_hits) * per_100
    return {
        'simile_density':          simile_hits * per_100,
        'intensifier_density':     intensifier_hits * per_100,
        'decorative_adj_density':  decorative_hits * per_100,
        'hollow_phrase_density':   hollow_hits * per_100,
        'rhetorical_q_rate':       rhetorical_q / n_sents,
        'anaphora_proxy':          anaphora,
        'fig_lang_composite':      composite,
    }


fig_features = [
    'simile_density', 'intensifier_density', 'decorative_adj_density',
    'hollow_phrase_density', 'rhetorical_q_rate', 'anaphora_proxy',
    'fig_lang_composite',
]
fig_rows = [
    {'student_id': row.student_id, **figurative_features(row.essay, doc)}
    for row, doc in tqdm(zip(df.itertuples(), docs), total=len(df), desc='Figurative language')
]
fig_df = pd.DataFrame(fig_rows)
df = df.drop(columns=[c for c in fig_features if c in df.columns])
df = df.merge(fig_df, on='student_id')

print('Figurative / decorative language â€” mean by corpus:')
print(df.groupby('corpus')[fig_features].mean().round(3).to_string())


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
order = [c for c in CORPUS_PALETTE if c in df['corpus'].unique()]

for ax, feat in zip(axes, fig_features):
    valid = df.dropna(subset=[feat])
    sns.violinplot(
        data=valid, x='corpus', y=feat, ax=ax,
        palette=CORPUS_PALETTE, cut=0, inner='quartile',
        order=[c for c in order if c in valid['corpus'].unique()],
    )
    ax.set_title(feat.replace('_', ' ').title(), fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)

for ax in axes[len(fig_features):]:
    ax.set_visible(False)

plt.suptitle('Figurative & decorative language features by corpus', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

# Score correlation restricted to real essays
real_df = df[df['corpus'] == 'real'].dropna(subset=['score'] + fig_features)
if len(real_df) > 1:
    corrs = real_df[fig_features + ['score']].corr()['score'].drop('score').round(3)
    print('\nPearson r with reviewer score (real essays only):')
    print(corrs)


---

In [ ]:
import re

_PRON_1SG  = r'\b(i|me|my|mine|myself)\b'
_PRON_1PL  = r'\b(we|us|our|ours|ourselves)\b'
_PRON_2ND  = r'\b(you|your|yours|yourself|yourselves)\b'
_PRON_3SG  = r'\b(he|him|his|she|her|hers|herself|himself|it|its|itself)\b'
_PRON_3PL  = r'\b(they|them|their|theirs|themselves)\b'
_PRON_REFL = r'\b(myself|yourself|yourselves|himself|herself|itself|ourselves|themselves|oneself)\b'
_SENT_RE   = re.compile(r'(?<=[.!?])\s+')

def pronoun_features(text):
    if not isinstance(text, str) or not text.strip():
        keys = ['pron_1sg_rate','pron_1pl_rate','pron_2nd_rate',
                'pron_3sg_rate','pron_3pl_rate','pron_refl_rate',
                'pron_i_rate','pron_we_rate',
                'pron_1sg_to_1pl_ratio','sent_start_i_rate']
        return {k: float('nan') for k in keys}
    words  = re.findall(r'\b\w+\b', text)
    n_w    = max(len(words), 1)
    scale  = 100 / n_w
    n_1sg  = len(re.findall(_PRON_1SG,  text, re.I))
    n_1pl  = len(re.findall(_PRON_1PL,  text, re.I))
    n_2nd  = len(re.findall(_PRON_2ND,  text, re.I))
    n_3sg  = len(re.findall(_PRON_3SG,  text, re.I))
    n_3pl  = len(re.findall(_PRON_3PL,  text, re.I))
    n_refl = len(re.findall(_PRON_REFL, text, re.I))
    n_i    = len(re.findall(r'\bI\b',   text))
    n_we   = len(re.findall(r'\bwe\b',  text, re.I))
    sents  = [s.strip() for s in _SENT_RE.split(text) if s.strip()]
    n_s    = max(len(sents), 1)
    si     = sum(1 for s in sents if re.match(r'^I\b', s))
    denom  = n_1sg + n_1pl
    return {
        'pron_1sg_rate':         n_1sg  * scale,
        'pron_1pl_rate':         n_1pl  * scale,
        'pron_2nd_rate':         n_2nd  * scale,
        'pron_3sg_rate':         n_3sg  * scale,
        'pron_3pl_rate':         n_3pl  * scale,
        'pron_refl_rate':        n_refl * scale,
        'pron_i_rate':           n_i    * scale,
        'pron_we_rate':          n_we   * scale,
        'pron_1sg_to_1pl_ratio': n_1sg / denom if denom > 0 else float('nan'),
        'sent_start_i_rate':     si / n_s,
    }

pron_rows = [
    {'student_id': row.student_id, **pronoun_features(row.essay)}
    for row in tqdm(df.itertuples(), total=len(df), desc='Pronoun features')
]
pron_df_  = pd.DataFrame(pron_rows)
pron_cols = [c for c in pron_df_.columns if c != 'student_id']
df = df.drop(columns=[c for c in pron_cols if c in df.columns], errors='ignore')
df = df.merge(pron_df_, on='student_id')

print('Pronoun features — mean by corpus:')
print(df.groupby('corpus')[pron_cols].mean().round(3).to_string())


In [ ]:
print(pron_cols)

In [ ]:
# ── Pronoun usage — distributions by corpus ───────────────────────────────────
plot_cols = ['pron_i_rate','pron_we_rate','pron_2nd_rate',
             'pron_1sg_to_1pl_ratio','sent_start_i_rate',
             'pron_3sg_rate','pron_3pl_rate','pron_refl_rate']
plot_cols = [c for c in plot_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
corpus_colors = {'real': '#3498db', 'synthetic': '#e74c3c', 'freeform': '#2ecc71'}

for ax, col in zip(axes, plot_cols):
    for corpus, grp in df.groupby('corpus'):
        vals = grp[col].dropna()
        ax.hist(vals, bins=40, alpha=0.55, label=corpus,
                color=corpus_colors.get(corpus, 'grey'), density=True)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('per 100 words' if 'rate' in col and 'sent_start' not in col and 'ratio' not in col else '')
    ax.tick_params(labelsize=8)

for ax in axes[len(plot_cols):]:
    ax.set_visible(False)

handles = [plt.Rectangle((0,0),1,1, color=c, alpha=0.6) for c in corpus_colors.values()]
fig.legend(handles, corpus_colors.keys(), loc='lower right', fontsize=9)
fig.suptitle('Pronoun Usage by Corpus', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

display(df.groupby('corpus')[plot_cols].mean().round(3))


In [ ]:
plot_cols = ['pron_i_rate','pron_we_rate','pron_2nd_rate',
             'pron_1sg_to_1pl_ratio','sent_start_i_rate',
             'pron_3sg_rate','pron_3pl_rate','pron_refl_rate']
plot_cols = [c for c in plot_cols if c in df.columns]

corpus_colors = {'real': '#3498db', 'synthetic': '#e74c3c', 'freeform': '#2ecc71'}
corpora_order = ['real', 'synthetic', 'freeform']
corpora_present = [c for c in corpora_order if c in df['corpus'].unique()]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, plot_cols):
    data = [df.loc[df['corpus'] == c, col].dropna().values for c in corpora_present]
    parts = ax.violinplot(data, positions=range(len(corpora_present)),
                          showmedians=True, showextrema=False)
    for i, (pc, corpus) in enumerate(zip(parts['bodies'], corpora_present)):
        pc.set_facecolor(corpus_colors[corpus])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(1.5)
    ax.set_xticks(range(len(corpora_present)))
    ax.set_xticklabels(corpora_present, fontsize=8)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=8)

for ax in axes[len(plot_cols):]:
    ax.set_visible(False)

handles = [plt.Rectangle((0,0),1,1, color=corpus_colors[c], alpha=0.7) for c in corpora_present]
fig.legend(handles, corpora_present, loc='lower right', fontsize=9)
fig.suptitle('Pronoun Usage by Corpus', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

display(df.groupby('corpus')[plot_cols].mean().round(3))


---
## Section 6: Style Clustering

Combine all stylometric features â€” including the new figurative language metrics â€”
into a single scaled feature matrix, then cluster essays into writer-style profiles
using KMeans. PCA projects the high-dimensional space to 2D for visualization,
coloured by both **style cluster** and **corpus** to reveal whether corpora
separate stylometrically.

Key question: **do the synthetic / freeform corpora occupy different regions of
style space than real essays?** If so, stylometric features may discriminate
LLM-generated writing without relying on content at all.


In [ ]:
all_style_features = pron_cols + (
    lex_features
    + syn_features
    + surf_features
    + marker_cols
    + fig_features
)

print(all_style_features)

# Drop only rows missing feature values â€” freeform essays have no score but
# are perfectly valid for clustering, so we keep them (score stays NaN).
style_df = df.dropna(subset=all_style_features)
print(f'{len(style_df):,} essays after dropping NaN feature rows '
      f'(removed {len(df) - len(style_df)})')
print(style_df.groupby('corpus').size().rename('n_essays'))

X = style_df[all_style_features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Choose k with elbow plot
inertias = []
k_range  = range(2, 10)
for k in tqdm(k_range, desc='KMeans elbow'):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(k_range), inertias, marker='o', linewidth=2)
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow plot â€” style feature KMeans')
plt.tight_layout()
plt.show()


In [ ]:
N_STYLE_CLUSTERS = 4   # adjust after inspecting the elbow plot

km_style = KMeans(n_clusters=N_STYLE_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
style_df = style_df.copy()
style_df['style_cluster'] = km_style.fit_predict(X_scaled)
df = df.merge(style_df[['student_id', 'style_cluster']], on='student_id', how='left')

# Score by style cluster
score_style = (
    style_df.groupby('style_cluster')['score']
    .agg(['mean', 'std', 'count'])
    .round(3)
)
score_style.columns = ['mean_score', 'std_score', 'n_essays']
print(score_style)

In [ ]:
# PCA to 2D for visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
style_df = style_df.copy()
style_df['pc1'] = X_pca[:, 0]
style_df['pc2'] = X_pca[:, 1]

# Cluster feature profiles â€” what makes each cluster distinctive?
cluster_means = (
    style_df.groupby('style_cluster')[all_style_features].mean()
)
# Z-score to show relative standing
cluster_z = (cluster_means - cluster_means.mean()) / cluster_means.std()

# Top 3 distinguishing features per cluster
print('\nTop distinguishing features per cluster (by z-score magnitude):')
for cluster_id, row in cluster_z.iterrows():
    top = row.abs().nlargest(3)
    desc = ', '.join(f'{f} ({row[f]:+.2f})' for f in top.index)
    mean_sc = score_style.loc[cluster_id, 'mean_score']
    print(f'  Cluster {cluster_id} (mean score {mean_sc:.2f}): {desc}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: PCA coloured by style cluster
cluster_palette = sns.color_palette('tab10', N_STYLE_CLUSTERS)
for cid in range(N_STYLE_CLUSTERS):
    mask = style_df['style_cluster'] == cid
    n = mask.sum()
    axes[0].scatter(
        style_df.loc[mask, 'pc1'],
        style_df.loc[mask, 'pc2'],
        color=cluster_palette[cid], alpha=0.5, edgecolors='none',
        label=f'Cluster {cid} (n={n})'
    )
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)')
axes[0].set_title('Style clusters in PCA space')
axes[0].legend(fontsize=8)

# Middle: PCA coloured by corpus
for corpus_name, colour in CORPUS_PALETTE.items():
    mask = style_df['corpus'] == corpus_name
    if mask.any():
        axes[1].scatter(
            style_df.loc[mask, 'pc1'],
            style_df.loc[mask, 'pc2'],
            color=colour, alpha=0.4, edgecolors='none',
            label=f'{corpus_name} (n={mask.sum()})'
        )
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)')
axes[1].set_title('Corpora in PCA style space')
axes[1].legend(fontsize=8)

# Right: Score distribution by style cluster (real essays only)
real_style = style_df[style_df['corpus'] == 'real'].dropna(subset=['score'])
if len(real_style) > 0:
    real_style.boxplot(column='score', by='style_cluster', ax=axes[2])
    axes[2].set_title('Score by style cluster (real essays)')
    axes[2].set_xlabel('Style cluster')
    axes[2].set_ylabel('Score')
else:
    axes[2].set_title('No real essays with scores available')
    axes[2].set_visible(False)

plt.suptitle('')
plt.tight_layout()
plt.show()


---
## Section 7: Feature correlation heatmap & export


In [ ]:
# Correlation matrix â€” real essays only (score is meaningful there)
real_df = df[df['corpus'] == 'real'].copy()
numeric_cols = all_style_features + (['score'] if 'score' in real_df.columns else [])
corr_matrix  = real_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, ax=ax, linewidths=0.3, annot_kws={'size': 6}
)
ax.set_title('Stylometric feature correlation matrix (real essays)', fontsize=12)
plt.tight_layout()
plt.show()

if 'score' in corr_matrix.columns:
    score_corr = corr_matrix['score'].drop('score').abs().sort_values(ascending=False)
    print('\nAll features ranked by |correlation| with score (real essays):')
    print(score_corr.round(3).to_string())


In [ ]:
output_dir = NOTEBOOK_DIR / 'outputs/stylometrics/'
output_dir.mkdir(parents=True, exist_ok=True)

pron_cols_export = [c for c in df.columns if c.startswith('pron_') or c == 'sent_start_i_rate']
print(pron_cols_export)
export_cols = ['student_id', 'score', 'style_cluster'] + pron_cols_export + all_style_features
df[[c for c in export_cols if c in df.columns]].to_csv(
    output_dir / 'stylometric_features.csv', index=False
)
print('Saved to:', output_dir)

---
## Section 8: Feature divergence â€” real vs. synthetic essays

Rank every stylometric feature by how well it separates **real** from **synthetic** essays.
We use two complementary statistics:

* **Cohen's d** â€” standardised mean difference (effect size). Positive = synthetic > real.
* **Mann-Whitney U p-value** â€” non-parametric test; no normality assumption.

Features with a large |d| *and* a small p-value are the strongest candidates for an AI-detection signal.

In [ ]:
from pathlib import Path
import pandas as pd

_csv = Path('../outputs/stylometrics/stylometric_features.csv')
df = pd.read_csv(_csv)

_prefix_map = {'rea': 'real', 'syn': 'synthetic', 'fre': 'freeform'}
df['corpus'] = df['student_id'].str[:3].map(_prefix_map)

_meta_cols = {'student_id', 'score', 'style_cluster', 'corpus'}
all_style_features = [c for c in df.columns if c not in _meta_cols]
print(all_style_features)

print(f"Loaded {len(df):,} essays  |  {len(all_style_features)} features")
print(df.groupby('corpus').size().rename('n'))

In [ ]:
from scipy import stats
import warnings

real_df = df[df['corpus'] == 'real'][all_style_features].dropna(how='all', axis=1)
ai_df   = df[df['corpus'].isin(['synthetic', 'freeform'])][all_style_features].dropna(how='all', axis=1)
shared_feats = [f for f in all_style_features if f in real_df.columns and f in ai_df.columns]

rows = []
for feat in shared_feats:
    r = real_df[feat].dropna()
    s = ai_df[feat].dropna()
    if len(r) < 2 or len(s) < 2:
        continue
    pooled_std = ((r.std() ** 2 + s.std() ** 2) / 2) ** 0.5
    cohens_d   = (s.mean() - r.mean()) / pooled_std if pooled_std > 0 else 0.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _, pval = stats.mannwhitneyu(s, r, alternative='two-sided')
    rows.append({'feature': feat, 'cohens_d': cohens_d, 'pval': pval,
                 'mean_real': r.mean(), 'mean_ai': s.mean()})

divergence_df = (
    pd.DataFrame(rows)
      .assign(abs_d=lambda x: x['cohens_d'].abs())
      .sort_values('abs_d', ascending=False)
      .reset_index(drop=True)
)

print(f'Features analysed: {len(divergence_df)}')
print("\nTop 20 by |Cohen's d| (real vs. synthetic+freeform):")
print(divergence_df[['feature','cohens_d','pval','mean_real','mean_ai']]
      .head(25).to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
import matplotlib.pyplot as plt

top_n = 30
plot_df = divergence_df.head(top_n).sort_values('cohens_d')

colors = ['#e74c3c' if d > 0 else '#3498db' for d in plot_df['cohens_d']]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(plot_df['feature'], plot_df['cohens_d'], color=colors, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel("Cohen's d  (positive = synthetic > real)", fontsize=11)
ax.set_title(f'Top {top_n} stylometric features by effect size\n(real vs. synthetic essays)', fontsize=12)

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    label = f"p={row['pval']:.2e}" if row['pval'] >= 1e-4 else 'p<0.0001'
    x = row['cohens_d'] + (0.02 if row['cohens_d'] >= 0 else -0.02)
    ha = 'left' if row['cohens_d'] >= 0 else 'right'
    ax.text(x, bar.get_y() + bar.get_height() / 2, label,
            va='center', ha=ha, fontsize=7, color='#555555')

plt.tight_layout()
plt.show()

divergence_df.to_csv(
    NOTEBOOK_DIR / '../outputs/stylometrics/feature_divergence_real_vs_synthetic.csv',
    index=False
)
print('Saved feature_divergence_real_vs_synthetic.csv')

In [ ]:
import pandas as pd
from pathlib import Path

stylo = pd.read_csv(NOTEBOOK_DIR / '../outputs/stylometrics/stylometric_features.csv')

# Real essays with mean sentence length > 80
long_sent = stylo[
    (stylo['student_id'].str.startswith('rea_')) &
    (stylo['mean_sent_len'] > 60)
].sort_values('mean_sent_len', ascending=False)[['student_id', 'mean_sent_len', 'n_sentences', 'n_tokens']]

print(f'{len(long_sent)} essays with mean_sent_len > 80')
display(long_sent)

# Load the actual essay text and show the worst offenders
essays = pd.read_excel(NOTEBOOK_DIR / '../data/real/essays.xlsx',
                       usecols=['Campus ID', 'Essay Response'])
essays['student_id'] = 'rea_' + essays['Campus ID'].astype(str)

for _, row in long_sent.head(5).iterrows():
    text = essays.loc[essays['student_id'] == row['student_id'], 'Essay Response'].values
    if len(text):
        print(f"\n{'='*60}")
        print(f"student_id: {row['student_id']}  mean_sent_len: {row['mean_sent_len']:.1f}  n_sentences: {row['n_sentences']}")
        print(text[0][:800])

In [ ]:
    st.subheader("Feature Divergence: Real vs. AI-Generated (Cohen's d)")
    st.caption("Positive d = AI-generated > real. Features sorted by absolute effect size.")

    div_df = load_divergence().sort_values("cohens_d")
    div_df["color"] = div_df["cohens_d"].apply(lambda d: "#e74c3c" if d > 0 else "#3498db")

    hover = {"pval": ":.2e", "mean_real": ":.3f"}
    if "mean_ai" in div_df.columns:
        hover["mean_ai"] = ":.3f"
    elif "mean_synthetic" in div_df.columns:
        hover["mean_synthetic"] = ":.3f"

    fig2 = px.bar(
        div_df, x="cohens_d", y="feature", orientation="h",
        color="cohens_d",
        color_continuous_scale=[[0, "#3498db"], [0.5, "#f0f0f0"], [1, "#e74c3c"]],
        color_continuous_midpoint=0,
        hover_data=hover,
        labels={"cohens_d": "Cohen's d", "feature": "Feature"},
        height=600,
    )


In [ ]:
short_sent = stylo[
    (stylo['student_id'].str.startswith('rea_')) &
    (stylo['n_sentences'] < 5)
].sort_values('n_sentences')[['student_id', 'n_sentences', 'n_tokens', 'mean_sent_len']]

print(f'{len(short_sent)} essays with fewer than 5 sentences')
display(short_sent)

for _, row in short_sent.head(10).iterrows():
    text = essays.loc[essays['student_id'] == row['student_id'], 'Essay Response'].values
    if len(text):
        print(f"\n{'='*60}")
        print(f"student_id: {row['student_id']}  n_sentences: {row['n_sentences']}  n_tokens: {row['n_tokens']}")
        print(text[0][:800])